# Stage 2 - Clean and align the data

Builds the monthly datasets used by the next stages. Reads `data/raw/` (never changed) and writes to `data/processed/`:

- `macro_realtime_12m.csv`, `macro_realtime_3m.csv`: macro indicators as an investor could see them at each month-end (ALFRED vintages), as 12-month and 3-month changes
- `macro_revised_12m.csv`, `macro_revised_3m.csv`: the same indicators with today's revised values, same timing
- `macro_data_month.csv`: which month of data was the latest available at each month-end
- `returns_daily.csv`: daily log returns of the 4 ETFs
- `returns_monthly.csv`: monthly simple returns of the 4 ETFs
- `stock_bond_corr.csv`: SPY-TLT correlation of daily returns, one value per month

Run from top to bottom in a fresh kernel.

In [1]:
import numpy as np
import pandas as pd

import config
import helpers

## 1. Load the raw files

In [2]:
vintages = helpers.load_alfred_vintages()
revised = helpers.load_revised_macro()
prices = helpers.load_etf_prices("Adj Close")

print("vintages:", vintages.shape, "| revised:", revised.shape, "| prices:", prices.shape)
print("missing values in vintages:", vintages["value"].isna().sum())
print(vintages[vintages["value"].isna()][["series_id", "date", "realtime_start"]])

vintages: (44691, 5) | revised: (5852, 3) | prices: (8473, 4)
missing values in vintages: 36
      series_id        date realtime_start
2      CPIAUCSL  1990-01-01     1991-09-13
8      CPIAUCSL  1990-02-01     1991-10-17
14     CPIAUCSL  1990-03-01     1991-11-14
19     CPIAUCSL  1990-04-01     1991-12-13
24     CPIAUCSL  1990-05-01     1992-01-16
28     CPIAUCSL  1990-06-01     1992-02-19
33     CPIAUCSL  1990-07-01     1992-03-17
39     CPIAUCSL  1990-08-01     1992-04-10
44     CPIAUCSL  1990-09-01     1992-05-13
50     CPIAUCSL  1990-10-01     1992-06-12
56     CPIAUCSL  1990-11-01     1992-07-14
62     CPIAUCSL  1990-12-01     1992-08-13
67     CPIAUCSL  1991-01-01     1992-09-15
72     CPIAUCSL  1991-02-01     1992-10-15
77     CPIAUCSL  1991-03-01     1992-11-13
83     CPIAUCSL  1991-04-01     1992-12-13
88     CPIAUCSL  1991-05-01     1993-01-15
93     CPIAUCSL  1991-06-01     1993-02-18
100    CPIAUCSL  1991-07-01     1993-03-17
105    CPIAUCSL  1991-08-01     1993-04-09
110 

## 2. Choices

- **Transformations.** Levels trend over time, so each indicator becomes a change, computed two ways:
  over 12 months (main version: smooth, shows the trend) and over 3 months, annualised (robustness check: faster but noisier).
  Log change in % for indices and quantities, change in points for the unemployment rate,
  level for capacity utilisation (already a rate that moves around a mean).
- **Initial claims (ICSA)** are weekly: they become the monthly average of the weeks, complete months only.
  Claims are almost never revised (median revision of the 12-month change after 2009: about 0.5 points).
- **Before a series' first vintage**, ALFRED has no real-time data: the revised values are used, with a one-month delay.
  This affects only the training years before 2006 (core CPI, PPI and capacity utilisation until 1996, retail until 2001),
  except claims, whose vintages start in May 2009.
- **Timing.** A decision is taken at each month-end. Only data published by that day can be used.
- **Missing months** (October 2025, shutdown): the last available value is kept, as an investor would.
- **Sample.** Decision dates from January 1991 (12-month changes need one year of data from 1990) to August 2026,
  the last complete month before the freeze date (28 September 2026).

In [3]:
# indicator -> (column name, transformation)
TRANSFORMS = {
    "INDPRO":   ("indpro",   "log"),
    "TCU":      ("tcu",      "level"),
    "RRSFS":    ("retail",   "log"),
    "HOUST":    ("housing",  "log"),
    "PAYEMS":   ("payrolls", "log"),
    "UNRATE":   ("unrate",   "diff"),
    "ICSA":     ("claims",   "log"),
    "CPIAUCSL": ("cpi",      "log"),
    "CPILFESL": ("core_cpi", "log"),
    "PPIACO":   ("ppi",      "log"),
}
HORIZONS = [12, 3]   # months

# first ALFRED vintage of each series: before that date only revised data exists
FIRST_VINTAGE = vintages.groupby("series_id")["realtime_start"].min()

start = pd.Timestamp(config.OBS_START) + pd.DateOffset(months=12)   # 12-month changes need one year of data
decision_dates = pd.date_range(start, "2026-08-31", freq="ME")
print(len(decision_dates), "decision dates:", decision_dates[0].date(), "->", decision_dates[-1].date())
print("first vintage of each series:")
print(FIRST_VINTAGE.to_string())

428 decision dates: 1991-01-31 -> 2026-08-31
first vintage of each series:
series_id
CPIAUCSL    1990-02-21
CPILFESL    1996-12-12
HOUST       1990-02-15
ICSA        2009-05-28
INDPRO      1990-02-16
PAYEMS      1990-02-02
PPIACO      1996-12-11
RRSFS       2001-06-15
TCU         1996-11-15
UNRATE      1990-02-02


In [4]:
def transform(s, how, h):
    """Change over h months: log change in % (annualised), change in points, or level."""
    if how == "log":
        return 100 * (12 / h) * np.log(s / s.shift(h))
    if how == "diff":
        return s - s.shift(h)
    return s


def to_monthly(values, sid, t=None):
    """Monthly series indexed by the first day of the month. ICSA: average of the weeks, complete months only."""
    values = values.copy()
    values.index = pd.to_datetime(values.index)
    if sid != "ICSA":
        return values.asfreq("MS")
    monthly = values.resample("MS").mean()
    if t is not None:                                    # the month of t is not complete yet
        monthly = monthly[monthly.index < t.to_period("M").to_timestamp()]
    return monthly

## 3. Real-time panels

For each decision date `t` and each indicator:
1. keep the vintage that was valid on `t` (`realtime_start <= t <= realtime_end`): the data as published that day;
2. apply the transformations to that vintage;
3. take the latest month with a value, and save which month it is.

Before a series' first vintage, the revised data is used instead, cut at the month before `t`.

In [5]:
revised_monthly = {sid: to_monthly(revised[revised["series_id"] == sid].set_index("date")["value"], sid)
                   for sid in TRANSFORMS}


def realtime_values(vint, sid, how, t):
    """Latest transformed values of one indicator, using only data published by date t."""
    ts = t.strftime("%Y-%m-%d")
    if ts < FIRST_VINTAGE[sid]:
        # no vintage yet: revised data, up to the month before t (the usual publication delay)
        s = revised_monthly[sid]
        s = s[s.index < t.to_period("M").to_timestamp()]
    else:
        snap = vint[(vint["realtime_start"] <= ts) & (vint["realtime_end"] >= ts)]
        if snap.empty:                                   # nothing published yet (RRSFS before June 2001)
            return {h: np.nan for h in HORIZONS}, pd.NaT
        s = to_monthly(snap.set_index("date")["value"], sid, t)

    x12 = transform(s, how, 12).dropna()
    if x12.empty:
        return {h: np.nan for h in HORIZONS}, pd.NaT
    month = x12.index[-1]                                # latest month with a value (skips October 2025)
    # ffill: a change that needs October 2025 keeps the last value, as in the revised panel
    return {h: transform(s, how, h).ffill().get(month, np.nan) for h in HORIZONS}, month


rt = {h: {} for h in HORIZONS}
months = {}
for sid, (name, how) in TRANSFORMS.items():
    vint = vintages[vintages["series_id"] == sid]
    out = [realtime_values(vint, sid, how, t) for t in decision_dates]
    for h in HORIZONS:
        rt[h][name] = [vals[h] for vals, m in out]
    months[name] = [m for vals, m in out]

macro_rt = {h: pd.DataFrame(rt[h], index=decision_dates).rename_axis("date") for h in HORIZONS}
data_month = pd.DataFrame(months, index=decision_dates).rename_axis("date")
macro_rt[12].tail()

,indpro,tcu,retail,housing,payrolls,unrate,claims,cpi,core_cpi,ppi
date,,,,,,,,,,
2026-04-30,0.738947,75.6596,0.659626,10.299610,0.164031,0.1,-6.963369,3.233124,2.568887,5.850790
2026-05-31,1.344065,76.1194,1.048692,4.538301,0.158249,0.1,-8.309276,3.709582,2.706355,9.366510
2026-06-30,1.652191,76.1663,2.568241,-9.089790,0.316852,0.0,-9.117723,4.082150,2.783831,12.289430
2026-07-31,1.137590,76.0937,3.097736,3.421574,0.318779,0.1,-7.153645,3.404900,2.533421,9.631093
2026-08-31,1.073671,76.2878,1.637292,-14.476747,0.199118,-0.2,-8.598584,3.250452,2.436590,7.946494


In [6]:
# how old is the information at each decision date (months between data month and decision date)
age = pd.DataFrame({c: (decision_dates.year - data_month[c].dt.year) * 12
                       + (decision_dates.month - data_month[c].dt.month) for c in data_month},
                   index=decision_dates)
print("data age in months (median / max):")
print(age.agg(["median", "max"]).T)

# months where revised data replaces the missing vintages
fallback = pd.DataFrame({TRANSFORMS[sid][0]: decision_dates.strftime("%Y-%m-%d") < FIRST_VINTAGE[sid]
                         for sid in TRANSFORMS}, index=decision_dates)
print("\nmonths using revised data instead of vintages:")
print(pd.DataFrame({"before 2006": fallback.loc[:"2005"].sum(), "from 2006": fallback.loc["2006":].sum()}))

data age in months (median / max):
          median  max
indpro       1.0  3.0
tcu          1.0  3.0
retail       1.0  3.0
housing      1.0  4.0
payrolls     1.0  2.0
unrate       1.0  2.0
claims       1.0  1.0
cpi          1.0  2.0
core_cpi     1.0  2.0
ppi          1.0  3.0

months using revised data instead of vintages:
          before 2006  from 2006
indpro              0          0
tcu                70          0
retail            125          0
housing             0          0
payrolls            0          0
unrate              0          0
claims            180         40
cpi                 0          0
core_cpi           71          0
ppi                71          0


## 4. Revised panels, same timing

Same transformations on today's revised data, read at the **same data month** as the real-time panel.
So the two panels differ only because of revisions, not because of timing.

In [7]:
macro_rev = {}
for h in HORIZONS:
    cols = {}
    for sid, (name, how) in TRANSFORMS.items():
        x = transform(revised_monthly[sid], how, h).ffill()   # October 2025 gap: keep the last value
        cols[name] = x.reindex(data_month[name]).to_numpy()
    macro_rev[h] = pd.DataFrame(cols, index=decision_dates).rename_axis("date")

for h in HORIZONS:
    print(f"\n{h}-month changes, mean absolute difference real time vs revised (from 2006):")
    print((macro_rt[h] - macro_rev[h]).loc["2006":].abs().mean().round(2).to_string())


12-month changes, mean absolute difference real time vs revised (from 2006):
indpro      1.15
tcu         1.03
retail      0.61
housing     2.32
payrolls    0.23
unrate      0.05
claims      1.31
cpi         0.03
core_cpi    0.02
ppi         0.14

3-month changes, mean absolute difference real time vs revised (from 2006):
indpro       2.02
tcu          1.03
retail       2.05
housing     10.35
payrolls     0.41
unrate       0.08
claims      10.24
cpi          0.51
core_cpi     0.32
ppi          0.73


## 5. ETF returns

- Daily log returns from Adjusted Close (dividends included), kept for covariance estimates later.
- Monthly simple returns from the last price of each month.
- DBC starts in February 2006, so all 4 assets have returns from March 2006.
- September 2026 is incomplete (data stop on the 28th) and is dropped.

In [8]:
last_month_end = decision_dates[-1]

ret_daily = np.log(prices).diff().loc[:last_month_end]
ret_daily = ret_daily.dropna(how="all")

month_prices = prices.resample("ME").last().loc[:last_month_end]
ret_monthly = month_prices.pct_change(fill_method=None).dropna(how="all")

common = ret_monthly.dropna().index
print("monthly returns, all 4 assets:", common[0].date(), "->", common[-1].date(), f"({len(common)} months)")
ret_monthly.describe().round(4)

monthly returns, all 4 assets: 2006-03-31 -> 2026-08-31 (246 months)


ticker,DBC,GLD,SPY,TLT
count,246.0000,261.0000,403.0000,289.0000
mean,0.0036,0.0097,0.0095,0.0036
std,0.0543,0.0498,0.0426,0.0389
min,-0.2508,-0.1614,-0.1652,-0.1307
25%,-0.0278,-0.0214,-0.0149,-0.0225
50%,0.0063,0.0056,0.0136,0.0034
75%,0.0405,0.0430,0.0367,0.0247
max,0.1627,0.1279,0.1270,0.1434


## 6. Stock-bond correlation, one value per month

Correlation of daily SPY and TLT returns inside each month (about 21 days).
This is the variable the regimes should explain.
Months with fewer than 15 days of both returns are left out (TLT starts at the end of July 2002).

In [9]:
def month_corr(g):
    g = g[["SPY", "TLT"]].dropna()
    return g["SPY"].corr(g["TLT"]) if len(g) >= 15 else np.nan


sb_corr = ret_daily.groupby(ret_daily.index.to_period("M")).apply(month_corr).dropna()
sb_corr.index = sb_corr.index.to_timestamp("M")
sb_corr = sb_corr.rename("spy_tlt_corr").to_frame()
sb_corr.index.name = "date"

print("average correlation by year:")
print(sb_corr["spy_tlt_corr"].groupby(sb_corr.index.year).mean().round(2).to_string())

average correlation by year:
date
2002   -0.64
2003   -0.31
2004   -0.04
2005    0.02
2006    0.06
2007   -0.34
2008   -0.52
2009   -0.35
2010   -0.48
2011   -0.59
2012   -0.66
2013   -0.24
2014   -0.45
2015   -0.38
2016   -0.31
2017   -0.34
2018   -0.27
2019   -0.40
2020   -0.42
2021   -0.13
2022    0.06
2023    0.14
2024    0.08
2025    0.05
2026    0.36


## 7. Checks and save

In [10]:
# the macro panels have the same dates, and every month with returns has a decision date
for h in HORIZONS:
    assert macro_rt[h].index.equals(macro_rev[h].index)
assert ret_monthly.loc["2001":].index.isin(decision_dates).all()
assert sb_corr.index.isin(decision_dates).all()

# no missing macro values from 2006 on (the asset sample)
for h in HORIZONS:
    print(f"missing values from 2006, {h}m: real time {int(macro_rt[h].loc['2006':].isna().sum().sum())},"
          f" revised {int(macro_rev[h].loc['2006':].isna().sum().sum())}")

outputs = {"macro_realtime_12m": macro_rt[12], "macro_revised_12m": macro_rev[12],
           "macro_realtime_3m": macro_rt[3], "macro_revised_3m": macro_rev[3],
           "macro_data_month": data_month, "returns_daily": ret_daily,
           "returns_monthly": ret_monthly, "stock_bond_corr": sb_corr}
for name, df in outputs.items():
    path = helpers.save_processed(df, name)
    print(f"saved {path.name}: {df.shape[0]} rows x {df.shape[1]} columns")

missing values from 2006, 12m: real time 0, revised 0
missing values from 2006, 3m: real time 0, revised 0
saved macro_realtime_12m.csv: 428 rows x 10 columns
saved macro_revised_12m.csv: 428 rows x 10 columns
saved macro_realtime_3m.csv: 428 rows x 10 columns
saved macro_revised_3m.csv: 428 rows x 10 columns
saved macro_data_month.csv: 428 rows x 10 columns
saved returns_daily.csv: 8453 rows x 4 columns
saved returns_monthly.csv: 403 rows x 4 columns
saved stock_bond_corr.csv: 289 rows x 1 columns
